# Premotor UMAP: raw versus baseline-subtracted
One session, four independent 3D embeddings: tracking/playback in columns and raw/baseline-subtracted activity in rows. Two independent figures show the same saved coordinates: **starting-position colors** and **direction colors** (left blue, right red, unknown gray). Both use larger circular points. Missing positions are gray in the position view.

Each point is one of five 100 ms premotor windows. All recorded neurons are included. No z-scoring, smoothing, PCA reduction, displacement filter or behavioral supervision is applied. Baseline subtraction uses each bout/neuron's own mean rate in **[-1, -0.7) s**.

Run top to bottom using the shared `.venv` kernel. Edit the settings and rerun below them to compare sessions, **15 / 30 / 60 neighbors**, or seeds. Embeddings use identical bouts in both activity modes. Separate panel axes have no common basis; compare patterns, not panel coordinates. UMAP prioritizes neighborhoods, and separated islands or a single attractive plot do not establish biological clusters. Multiple windows from one bout are dependent.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
repo_dir = next((candidate for parent in (cwd, *cwd.parents)
                 for candidate in (parent, parent / "etheremin_project")
                 if (candidate / "movement_bouts" / "utils.py").is_file()), None)
if repo_dir is None:
    raise FileNotFoundError("Run from the eTheremin workspace or inside the repository.")
sys.path.insert(0, str(repo_dir / "movement_bouts"))
from utils import prepare_pca_input
from umap_analysis import fit_umap_by_condition, plot_umap_comparison, make_interactive_umap_comparison

bout_dir = repo_dir.parent / "data" / "derived" / "movement_bouts"
premotor_dir = repo_dir.parent / "data" / "derived" / "premotor_activity"


## Settings
`N_NEIGHBORS=30` is the starting view, with 15 and 60 as sensitivity checks. Hold the other parameters fixed while comparing neighborhood sizes; then try other seeds. If a condition has too few observations, it is explicitly skipped rather than silently changing the neighborhood size. Raw and corrected panels use the same UMAP settings and seed.


In [ ]:
SESSION_IDX = 0
N_NEIGHBORS = 30
MIN_DIST = 0.1
RANDOM_STATE = 42
ANALYSIS_WINDOW_S = (-0.5, 0.0)
BASELINE_WINDOW_S = (-1.0, -0.7)
N_BINS = 5
EXTRACTION_SUBDIR = "extracted_-1.2_0"
POINT_SIZE = 5  # Interactive marker size in pixels.
SAVE_RESULTS = True


## Load the saved extraction
Verify that the bout/session tables match the extraction fingerprints. Loading the saved counts requires roughly 0.9 GB; only the selected session is retained below. Existing PCA caches are not needed.


In [ ]:
extraction_dir = premotor_dir / EXTRACTION_SUBDIR
extraction_parameters = json.loads((extraction_dir / "parameters.json").read_text(encoding="utf-8"))
for filename, expected_hash in extraction_parameters["source_sha256"].items():
    if hashlib.sha256((bout_dir / filename).read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f"{filename} has changed since extraction; regenerate aligned counts first.")
movement_bouts = pd.read_pickle(bout_dir / "movement_bouts.pkl")
session_info = pd.read_pickle(bout_dir / "session_info.pkl")
neuron_ids_by_session = pd.read_pickle(extraction_dir / "neuron_ids_by_session.pkl")
if SESSION_IDX not in session_info.index:
    raise ValueError(f"Unknown session idx={SESSION_IDX}.")
bout_ids = movement_bouts.index[movement_bouts["idx"].eq(SESSION_IDX)]
neuronal_bouts = pd.read_pickle(extraction_dir / "neuronal_bouts.pkl")
neuronal_bouts = neuronal_bouts.loc[neuronal_bouts.index.intersection(bout_ids)].copy()
if neuronal_bouts.empty:
    raise ValueError("No complete extracted bouts for this session.")
display(session_info.loc[[SESSION_IDX]])
print(f"{len(neuronal_bouts):,} complete bouts; {len(bout_ids) - len(neuronal_bouts):,} without extraction.")


## Fit the four panels
The first fit can take longer while Numba compiles. `umap_bundle` retains models, indexed labeled coordinates, exact neuron order, fit counts and preprocessing/extraction provenance. No behavior label is passed to UMAP.


In [ ]:
settings = {
    "idx": SESSION_IDX, "n_neighbors": N_NEIGHBORS, "min_dist": MIN_DIST,
    "random_state": RANDOM_STATE, "n_components": 3, "metric": "euclidean",
    "init": "random", "compatibility_layout": True, "n_jobs": 1, "standardized": False,
    "DT": extraction_parameters["DT"], "N_BINS": N_BINS,
    "analysis_window_s": list(ANALYSIS_WINDOW_S),
    "baseline_window_s": list(BASELINE_WINDOW_S),
    "activity_modes": ["raw", "baseline_subtracted"],
    "basis": "separate_conditions_and_activity_modes", "neuron_population": "all",
    "extraction_directory": str(extraction_dir), "extraction_parameters": extraction_parameters,
    "umap_version": version("umap-learn"),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}
fits, summaries = {}, {}
reference_index = None
for mode in settings["activity_modes"]:
    data, labels = prepare_pca_input(
        neuronal_bouts, movement_bouts, neuron_ids_by_session,
        idx=SESSION_IDX, dt=settings["DT"], n_bins=N_BINS,
        extraction_window_s=extraction_parameters["window_s"],
        analysis_window_s=ANALYSIS_WINDOW_S, baseline_window_s=BASELINE_WINDOW_S,
        activity_mode=mode,
    )
    if reference_index is None:
        reference_index = data.index.copy()
    elif not reference_index.equals(data.index):
        raise ValueError("Raw/corrected observation indices differ.")
    fits[mode], summaries[mode] = fit_umap_by_condition(
        data, labels, n_neighbors=N_NEIGHBORS, min_dist=MIN_DIST, random_state=RANDOM_STATE,
    )
    print(f"{mode}: {len(fits[mode])} condition embeddings fitted.")
umap_bundle = {
    "settings": settings, "fits": fits,
    "fit_summary": pd.concat(summaries, names=["activity_mode"]),
    "session_info": session_info.loc[[SESSION_IDX]].copy(),
    "position_limits": (float(movement_bouts["initial_position"].min()),
                        float(movement_bouts["initial_position"].max())),
}
display(umap_bundle["fit_summary"])


## Save the shared embeddings
Both views reuse `umap_bundle`; changing the coloring does not refit UMAP. The two figures retain independent rotation controls. Re-run either plotting cell to recolor the saved coordinates.


In [ ]:
if SAVE_RESULTS:
    run_dir = premotor_dir / "runs" / "umap_raw_vs_baseline"
    run_dir.mkdir(parents=True, exist_ok=True)
    effective_settings = {k: v for k, v in settings.items() if k != "created_at_utc"}
    digest = hashlib.sha256(json.dumps(effective_settings, sort_keys=True).encode()).hexdigest()[:12]
    stem = f"umap3d_session{SESSION_IDX}_k{N_NEIGHBORS}_seed{RANDOM_STATE}_{digest}"
    bundle_path = run_dir / f"{stem}.pkl"
    pd.to_pickle(umap_bundle, bundle_path)
    print(f"Saved {bundle_path}")


## Starting position
Starting position sets color on the fixed positional scale. All points are circles; direction is available on hover. Drag to rotate and scroll to zoom.


In [ ]:
position_fig = make_interactive_umap_comparison(umap_bundle, color_by="position", point_size=POINT_SIZE)
if SAVE_RESULTS:
    position_fig.write_html(run_dir / f"{stem}_position.html", include_plotlyjs=True)
    fig, axes = plot_umap_comparison(umap_bundle, color_by="position")
    fig.savefig(run_dir / f"{stem}_position.png", dpi=180, bbox_inches="tight")
    plt.close(fig)
display(position_fig)


## Initial movement direction
**Blue = leftward, red = rightward, gray = unknown.** Direction describes movement, not starting side. All points are circles, and these are the same embeddings and observations as the position view.


In [ ]:
direction_fig = make_interactive_umap_comparison(umap_bundle, color_by="direction", point_size=POINT_SIZE)
if SAVE_RESULTS:
    direction_fig.write_html(run_dir / f"{stem}_direction.html", include_plotlyjs=True)
    fig, axes = plot_umap_comparison(umap_bundle, color_by="direction")
    fig.savefig(run_dir / f"{stem}_direction.png", dpi=180, bbox_inches="tight")
    plt.close(fig)
display(direction_fig)


## Overall fits across sessions
Fit separate UMAPs for **all saved sessions** and both activity modes, then reuse the PCA evaluation methods on `UMAP1`?`UMAP3`: ordinary linear regression for starting/final position, and balanced L2 logistic regression for initial direction. Pool all five windows per session/condition; count bouts separately. No displacement filter for starting position or direction. Final-position filtering is optional and defaults to none.

Scores are **descriptive and in-sample**, not tests of significance or held-out decoding. Adjusted R? uses observation-row counts and does not account for dependence between a bout's windows. UMAP can change across random seeds; these fits describe this realization. Raw and corrected results are separate. Session entry is the pairing unit, not independent animals.

This section runs independently of the single-session browser cells after imports/settings. It loads the extraction once. Session bundles are saved as they finish; reuse requires matching settings. With `RANDOM_STATE=None`, each batch gets a unique run directory, and the exact fitted coordinates/models are saved.


In [ ]:
OVERALL_SESSION_INDICES = None  # None = all saved sessions; or e.g. [0, 1, 2].
EVALUATION_COLUMNS = ["UMAP1", "UMAP2", "UMAP3"]
DESTINATION_MIN_DISPLACEMENT = None  # Set 5.0 for abs(final-initial) > 5.
DIRECTION_C = 1.0
LINEAR_SCORE = "r2_adjusted"  # Or "r2".
REUSE_SAVED_EMBEDDINGS = True


In [ ]:
from importlib import reload
import evaluation
reload(evaluation)  # Refresh helpers when continuing in an existing notebook kernel.
from tqdm.auto import tqdm

extraction_dir = premotor_dir / EXTRACTION_SUBDIR
extraction_parameters = json.loads((extraction_dir / "parameters.json").read_text(encoding="utf-8"))
for filename, expected_hash in extraction_parameters["source_sha256"].items():
    if hashlib.sha256((bout_dir / filename).read_bytes()).hexdigest() != expected_hash:
        raise ValueError(f"{filename} has changed since extraction; regenerate aligned counts first.")
movement_bouts = pd.read_pickle(bout_dir / "movement_bouts.pkl")
session_info = pd.read_pickle(bout_dir / "session_info.pkl")
neuron_ids_by_session = pd.read_pickle(extraction_dir / "neuron_ids_by_session.pkl")
overall_indices = list(session_info.index if OVERALL_SESSION_INDICES is None else OVERALL_SESSION_INDICES)
if not overall_indices or len(set(overall_indices)) != len(overall_indices):
    raise ValueError("Choose a nonempty selection of unique session indices.")
if not pd.Index(overall_indices).isin(session_info.index).all():
    raise ValueError("Unknown overall session index.")
overall_settings = {
    "representation": "UMAP", "n_components": 3, "n_neighbors": N_NEIGHBORS,
    "min_dist": MIN_DIST, "random_state": RANDOM_STATE, "metric": "euclidean",
    "init": "random", "compatibility_layout": True, "n_jobs": 1,
    "DT": extraction_parameters["DT"], "N_BINS": N_BINS,
    "analysis_window_s": list(ANALYSIS_WINDOW_S), "baseline_window_s": list(BASELINE_WINDOW_S),
    "activity_modes": ["raw", "baseline_subtracted"], "standardized": False,
    "neuron_population": "all", "session_indices": [int(idx) for idx in overall_indices],
    "extraction_directory": str(extraction_dir), "extraction_parameters": extraction_parameters,
    "umap_version": version("umap-learn"),
}
overall_digest = hashlib.sha256(json.dumps(overall_settings, sort_keys=True).encode()).hexdigest()[:12]
if RANDOM_STATE is None:
    overall_digest += "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
overall_run_dir = premotor_dir / "runs" / f"umap_overall_k{N_NEIGHBORS}_seed{RANDOM_STATE}_{overall_digest}"
if SAVE_RESULTS:
    (overall_run_dir / "sessions").mkdir(parents=True, exist_ok=True)
    (overall_run_dir / "settings.json").write_text(json.dumps(overall_settings, indent=2))
overall_caches = {mode: {"sessions": {}, "settings": {**overall_settings, "activity_mode": mode},
                         "session_info": session_info.loc[overall_indices].copy(),
                         "representation": "UMAP", "projection_basis": "separate_condition_umaps"}
                  for mode in overall_settings["activity_modes"]}
overall_fit_rows = []
# Retain original source counts once; each session is averaged independently.
all_neuronal_bouts = pd.read_pickle(extraction_dir / "neuronal_bouts.pkl")
for idx in tqdm(overall_indices, desc="UMAP sessions"):
    session_path = overall_run_dir / "sessions" / f"session_{idx}.pkl"
    if REUSE_SAVED_EMBEDDINGS and session_path.exists():
        session_bundle = pd.read_pickle(session_path)
        if session_bundle["settings"] != overall_settings:
            raise ValueError(f"Saved session {idx} settings do not match this run.")
    else:
        session_fits, session_summaries = {}, {}
        ids = movement_bouts.index[movement_bouts["idx"].eq(idx)]
        has_counts = ids.isin(all_neuronal_bouts.index).any()
        reference_index = None
        for mode in overall_settings["activity_modes"]:
            if not has_counts:
                session_fits[mode] = {}
                session_summaries[mode] = pd.DataFrame([
                    {"condition": c, "status": "skipped", "reason": "No complete extracted bouts",
                     "n_observations": 0, "n_bouts": 0, "n_neurons": len(neuron_ids_by_session.loc[idx, "neuron_ids"]),
                     "n_neighbors": N_NEIGHBORS} for c in (0, 1)]).set_index("condition")
                continue
            data, labels = prepare_pca_input(
                all_neuronal_bouts, movement_bouts, neuron_ids_by_session, idx=idx,
                dt=overall_settings["DT"], n_bins=N_BINS,
                extraction_window_s=extraction_parameters["window_s"],
                analysis_window_s=ANALYSIS_WINDOW_S, baseline_window_s=BASELINE_WINDOW_S,
                activity_mode=mode,
            )
            if reference_index is not None and not reference_index.equals(data.index):
                raise ValueError(f"Raw/corrected indices differ for session {idx}.")
            reference_index = data.index.copy()
            session_fits[mode], session_summaries[mode] = fit_umap_by_condition(
                data, labels, n_neighbors=N_NEIGHBORS, min_dist=MIN_DIST, random_state=RANDOM_STATE,
            )
        session_bundle = {"settings": overall_settings, "fits": session_fits,
                          "fit_summary": pd.concat(session_summaries, names=["activity_mode"])}
        if SAVE_RESULTS:
            pd.to_pickle(session_bundle, session_path)
    for mode in overall_settings["activity_modes"]:
        overall_caches[mode]["sessions"][idx] = session_bundle["fits"][mode]
    summary = session_bundle["fit_summary"].reset_index()
    summary["idx"] = idx
    overall_fit_rows.append(summary)
del all_neuronal_bouts
overall_fit_summary = pd.concat(overall_fit_rows, ignore_index=True).set_index(["activity_mode", "idx", "condition"])
if SAVE_RESULTS:
    overall_fit_summary.to_pickle(overall_run_dir / "embedding_fit_summary.pkl")
display(overall_fit_summary.groupby(["activity_mode", "status"]).size().rename("n_embeddings"))
print(overall_run_dir)


## Starting-position regression
One fit per session and condition, using all windows pooled. Raw and baseline-subtracted summaries and matched tracking/playback plots are shown separately.


In [ ]:
from importlib import reload
import evaluation
reload(evaluation)  # Refresh cached helpers without refitting UMAP.

if "overall_evaluations" not in globals():
    overall_evaluations = {}
overall_evaluations["initial_position"] = {}
for mode, cache in overall_caches.items():
    if not any(cache["sessions"].values()):
        print(f"{mode}: no successful embeddings; inspect overall_fit_summary.")
        continue
    result = evaluation.run_label_regressions(cache, target="initial_position", columns=EVALUATION_COLUMNS, min_displacement=None)
    result["settings"]["saved_at_utc"] = datetime.now(timezone.utc).isoformat()
    result["embedding_fit_summary"] = overall_fit_summary.loc[mode].copy()
    overall_evaluations["initial_position"][mode] = result
    evaluation_settings = result["settings"]
    tag = hashlib.sha256(json.dumps({k: v for k, v in evaluation_settings.items() if k != "saved_at_utc"}, sort_keys=True).encode()).hexdigest()[:12]
    filename = f"initial_position_{mode}_{tag}_pooled"
    if SAVE_RESULTS:
        evaluation_dir = overall_run_dir / "evaluation"
        evaluation_dir.mkdir(parents=True, exist_ok=True)
        pd.to_pickle(result, evaluation_dir / f"{filename}.pkl")
    display(result["summary"])
    fig, axes = evaluation.plot_regression_summary(result["summary"], score=LINEAR_SCORE, representation="UMAP")
    fig.suptitle(f"Starting-position regression | {mode}", y=1.03)
    if SAVE_RESULTS:
        fig.savefig(evaluation_dir / f"{filename}_summary.png", dpi=160, bbox_inches="tight")
    plt.show()
    try:
        fig, axes, paired = evaluation.plot_paired_regressions(result["summary"], score=LINEAR_SCORE)
    except ValueError as error:
        print(f"{mode}: {error}")
    else:
        result["paired"] = paired
        fig.suptitle(f"Starting-position regression | {mode}", y=1.03)
        if SAVE_RESULTS:
            fig.savefig(evaluation_dir / f"{filename}_paired.png", dpi=160, bbox_inches="tight")
            pd.to_pickle(result, evaluation_dir / f"{filename}.pkl")
        plt.show()


## Final-position regression
One fit per session and condition, using all windows pooled. Raw and baseline-subtracted summaries and matched tracking/playback plots are shown separately.


In [ ]:
from importlib import reload
import evaluation
reload(evaluation)  # Refresh cached helpers without refitting UMAP.

if "overall_evaluations" not in globals():
    overall_evaluations = {}
overall_evaluations["final_position"] = {}
for mode, cache in overall_caches.items():
    if not any(cache["sessions"].values()):
        print(f"{mode}: no successful embeddings; inspect overall_fit_summary.")
        continue
    result = evaluation.run_label_regressions(cache, target="final_position", columns=EVALUATION_COLUMNS, min_displacement=DESTINATION_MIN_DISPLACEMENT)
    result["settings"]["saved_at_utc"] = datetime.now(timezone.utc).isoformat()
    result["embedding_fit_summary"] = overall_fit_summary.loc[mode].copy()
    overall_evaluations["final_position"][mode] = result
    evaluation_settings = result["settings"]
    tag = hashlib.sha256(json.dumps({k: v for k, v in evaluation_settings.items() if k != "saved_at_utc"}, sort_keys=True).encode()).hexdigest()[:12]
    filename = f"final_position_{mode}_{tag}_pooled"
    if SAVE_RESULTS:
        evaluation_dir = overall_run_dir / "evaluation"
        evaluation_dir.mkdir(parents=True, exist_ok=True)
        pd.to_pickle(result, evaluation_dir / f"{filename}.pkl")
    display(result["summary"])
    fig, axes = evaluation.plot_regression_summary(result["summary"], score=LINEAR_SCORE, representation="UMAP")
    fig.suptitle(f"Final-position regression | {mode}", y=1.03)
    if SAVE_RESULTS:
        fig.savefig(evaluation_dir / f"{filename}_summary.png", dpi=160, bbox_inches="tight")
    plt.show()
    try:
        fig, axes, paired = evaluation.plot_paired_regressions(result["summary"], score=LINEAR_SCORE)
    except ValueError as error:
        print(f"{mode}: {error}")
    else:
        result["paired"] = paired
        fig.suptitle(f"Final-position regression | {mode}", y=1.03)
        if SAVE_RESULTS:
            fig.savefig(evaluation_dir / f"{filename}_paired.png", dpi=160, bbox_inches="tight")
            pd.to_pickle(result, evaluation_dir / f"{filename}.pkl")
        plt.show()


## Direction logistic regression
One fit per session and condition, using all windows pooled. Raw and baseline-subtracted summaries and matched tracking/playback plots are shown separately.


In [ ]:
from importlib import reload
import evaluation
reload(evaluation)  # Refresh cached helpers without refitting UMAP.

if "overall_evaluations" not in globals():
    overall_evaluations = {}
overall_evaluations["initial_direction"] = {}
for mode, cache in overall_caches.items():
    if not any(cache["sessions"].values()):
        print(f"{mode}: no successful embeddings; inspect overall_fit_summary.")
        continue
    result = evaluation.run_direction_logistics(cache, columns=EVALUATION_COLUMNS, C=DIRECTION_C)
    result["settings"]["saved_at_utc"] = datetime.now(timezone.utc).isoformat()
    result["embedding_fit_summary"] = overall_fit_summary.loc[mode].copy()
    overall_evaluations["initial_direction"][mode] = result
    evaluation_settings = result["settings"]
    tag = hashlib.sha256(json.dumps({k: v for k, v in evaluation_settings.items() if k != "saved_at_utc"}, sort_keys=True).encode()).hexdigest()[:12]
    filename = f"initial_direction_{mode}_{tag}_pooled"
    if SAVE_RESULTS:
        evaluation_dir = overall_run_dir / "evaluation"
        evaluation_dir.mkdir(parents=True, exist_ok=True)
        pd.to_pickle(result, evaluation_dir / f"{filename}.pkl")
    display(result["summary"])
    fig, axes = evaluation.plot_regression_summary(result["summary"], score="balanced_accuracy", representation="UMAP")
    fig.suptitle(f"Direction logistic regression | {mode}", y=1.03)
    if SAVE_RESULTS:
        fig.savefig(evaluation_dir / f"{filename}_summary.png", dpi=160, bbox_inches="tight")
    plt.show()
    try:
        fig, axes, paired = evaluation.plot_paired_regressions(result["summary"], score="balanced_accuracy")
    except ValueError as error:
        print(f"{mode}: {error}")
    else:
        result["paired"] = paired
        fig.suptitle(f"Direction logistic regression | {mode}", y=1.03)
        if SAVE_RESULTS:
            fig.savefig(evaluation_dir / f"{filename}_paired.png", dpi=160, bbox_inches="tight")
            pd.to_pickle(result, evaluation_dir / f"{filename}.pkl")
        plt.show()
